In [ ]:
import scanpy as sc
import numpy as np
import flowkit as fk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from preprocessing import read_flow
fcs_dir = str(input("Path"))
tissue_type = input(str("Tissue type"))
df_flow, sample_list, session = read_flow(fcs_dir)

In [ ]:
df_flow['PD1'] = df_flow[['PD1', 'PD-1']].max(axis=1)
df_flow['TCRva'] = df_flow[['TCRva', 'TCRVaJa']].max(axis=1)
df_flow['41BB'] = df_flow[['41BB', '4-1BB']].max(axis=1)

In [ ]:
df_flow.drop(columns = ['PD-1', 'TCRVaJa', '4-1BB'], inplace=True)

In [ ]:
exclude = ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-B-A', 'SSC-B-H', 'SSC-H', 'AF-A', 'CD66bCD19CD326LD', 'Time', 'CD45']
df_flow = df_flow.drop(columns=exclude)

In [ ]:
df_flow_counts = df_flow.select_dtypes(include=[np.number])

In [ ]:
from preprocessing import pd_to_adata
adata = pd_to_adata(df_flow, df_flow_counts)

In [ ]:
sc.pp.scale(adata, max_value=5)

In [ ]:
adata

In [ ]:
adata_list = []
for group in ['ctr', 'hst', 'ftl']:
    group_data = adata[adata.obs['group'] == group].copy()
    n_cells = group_data.shape[0] 
    n_sample = min(20000, n_cells)
    sampled_indices = np.random.choice(n_cells, n_sample, replace=False)
    group_sample = group_data[sampled_indices]
    group_sample.obs['group'] = group
    adata_list.append(group_sample)

adata_filtered = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
CD3 = adata[(adata[:, 'CD3'].X > 0)]
CD3 = CD3[:, CD3.var.index != 'CD3']
adata_list = [
    CD3[CD3.obs['group'] == group].copy()[np.random.choice(
        CD3[CD3.obs['group'] == group].shape[0], 20000, replace=False
    )]
    for group in ['ctr', 'hst', 'ftl']
]

for i, group in enumerate(['ctr', 'hst', 'ftl']):
    adata_list[i].obs['group'] = group

CD3 = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
adata = adata[(adata[:, 'CD3'].X > 0)]
adata = adata[:, adata.var.index != 'CD3']
adata

In [ ]:
CD4 = adata[(adata[:, 'CD4'].X > 0)]
CD4 = CD4[:, CD4.var.index != 'gdTCR']
CD4 = CD4[:, CD4.var.index != 'TCRva']
CD4 = CD4[:, CD4.var.index != 'CD4']
CD4 = CD4[:, CD4.var.index != 'CD8']

adata_list = []
for group in ['ctr', 'hst', 'ftl']:
    group_data = CD4[CD4.obs['group'] == group].copy()
    n_cells = group_data.shape[0]
    n_sample = min(20000, n_cells)
    sampled_indices = np.random.choice(n_cells, n_sample, replace=False)
    group_sample = group_data[sampled_indices]
    group_sample.obs['group'] = group
    adata_list.append(group_sample)
    
CD4 = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
CD8 = adata[(adata[:, 'CD8'].X > 0)]
CD8 = CD8[:, CD8.var.index != 'gdTCR']
CD8 = CD8[:, CD8.var.index != 'TCRva']
CD8 = CD8[:, CD8.var.index != 'CD4']
CD8 = CD8[:, CD8.var.index != 'CD8']

adata_list = []
for group in ['ctr', 'hst', 'ftl']:
    group_data = CD8[CD8.obs['group'] == group].copy()
    n_cells = group_data.shape[0]
    n_sample = min(20000, n_cells)
    sampled_indices = np.random.choice(n_cells, n_sample, replace=False)
    group_sample = group_data[sampled_indices]
    group_sample.obs['group'] = group
    adata_list.append(group_sample)
    
CD8 = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
gdTCR = adata[(adata[:, 'gdTCR'].X > 0)]
gdTCR = gdTCR[:, gdTCR.var.index != 'gdTCR']
gdTCR = gdTCR[:, gdTCR.var.index != 'TCRva']

adata_list = []
for group in ['ctr', 'hst', 'ftl']:
    group_data = gdTCR[gdTCR.obs['group'] == group].copy()
    n_cells = group_data.shape[0]
    n_sample = min(20000, n_cells)
    sampled_indices = np.random.choice(n_cells, n_sample, replace=False)
    group_sample = group_data[sampled_indices]
    group_sample.obs['group'] = group
    adata_list.append(group_sample)
    
gdTCR = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
TCRva = adata[(adata[:, 'TCRva'].X > 0)]
TCRva = TCRva[:, TCRva.var.index != 'gdTCR']
TCRva = TCRva[:, TCRva.var.index != 'TCRva']

adata_list = []
for group in ['ctr', 'hst', 'ftl']:
    group_data = TCRva[TCRva.obs['group'] == group].copy()
    n_cells = group_data.shape[0]
    n_sample = min(20000, n_cells)
    sampled_indices = np.random.choice(n_cells, n_sample, replace=False)
    group_sample = group_data[sampled_indices]
    group_sample.obs['group'] = group
    adata_list.append(group_sample)
    
TCRva = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
sample = gdTCR
sample_name = 'gdTCR'

In [ ]:
sc.settings.verbosity = 3

In [ ]:
plt.rcParams.update({'font.size': 10})  

In [ ]:
sc.tl.pca(sample, svd_solver="arpack")
sc.pl.pca_variance_ratio(sample, log=False)

In [ ]:
sc.pl.pca_loadings(sample, components = '1,2')

In [ ]:
from preprocessing import pca_df
grouped_pca = pca_df(sample, session)

In [ ]:
from plotting_methods import pca_plot
pca_plot(grouped_pca, tissue_type, sample_name)

In [ ]:
sc.external.pp.harmony_integrate(sample, key='sample_id', max_iter_harmony=20)
sc.pp.neighbors(sample, use_rep='X_pca_harmony')
sc.tl.umap(sample)

In [ ]:
markers = list(sample.var_names)

In [ ]:
sc.pl.umap(sample, color= ['group'], cmap='turbo', title = '{} {} Groups'.format(tissue_type, sample_name))

In [ ]:
sc.tl.leiden(sample, resolution=0.6, flavor='leidenalg')

In [ ]:
sc.pl.umap(sample, color= ['leiden'], cmap='turbo', title = '{} {} Clusters'.format(tissue_type, sample_name))

In [ ]:
sc.pl.umap(sample, color= ['sample_id'], cmap='turbo',  title = '{} {} sample_id'.format(tissue_type, sample_name))

In [ ]:
plt.rcParams.update({'font.size': 14})  
sc.pl.umap(sample, color=markers, cmap='turbo')

In [ ]:
sc.tl.rank_genes_groups(sample, "leiden", method="wilcoxon")
df = pd.DataFrame(
    {group: sample.uns["rank_genes_groups"]["names"][group] for group in sample.uns["rank_genes_groups"]["names"].dtype.names}
)

In [ ]:
sc.pl.dotplot(sample, markers, swap_axes=True, groupby='leiden', title = "{} {} Dotplot".format(tissue_type, sample_name), cmap='RdBu_r', vmin = -4, vmax = 4)

In [ ]:
celltype = {'celltype': []}
cluster_to_genes = {
    '0': 'CD45RA+CD4+CCR7+ (0)',
    '1': 'CD45RA+CD8+ (1)',
    '2': 'CD45RA+CCR7+ (2)',
    '3': 'CD45RA+CCR7+IL33R+ (3)',
    '4': 'CD69+CD4+ (4)',
    '5': 'CD4+ (5)',
    '6': 'IL33R+ (6)',
    '7': 'FOXP3+CD25+CD4+CD28+IL33R+ (7)',
    '8': 'CXCR3+CD4+ (8)',
    '9': 'CD69+CD4+PD1+ (9)',
    '10': 'CXCR5+CD4+ (10)',
    '11': 'CD4+CD28+IL33R+ (11)',
    '12': 'CD45RA+CXCR6+IL33R+ (12)',
    '13': 'CD69+CD4+PD1+CXCR6+ (13)',
    '14': 'CD69+PD1+CD57+ (14)',
    '15': 'CD103+CXCR3+CD69+CD8+ (15)',
    '16': 'RORgt+ (16)',
    '17': 'CD4+CRTH2+ (17)'

}
celltype['celltype'] = [cluster_to_genes[leiden] for leiden in sample.obs['leiden']]
sample.obs["celltype"] = celltype['celltype']

print(sample.obs[["leiden", "celltype"]].head())

In [ ]:
# result = sample.uns["rank_genes_groups"]
# groups = result["names"].dtype.names

# celltype = {'celltype': []}
# cluster_to_genes = {}
# for group in groups:
#     top_genes = result["names"][group][:3]
#     cluster_to_genes[group] = f"{':'.join(top_genes)} ({group})"

# celltype['celltype'] = [cluster_to_genes[leiden] for leiden in sample.obs['leiden']]
# sample.obs["celltype"] = celltype['celltype']

# print(sample.obs[["leiden", "celltype"]].head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.umap(
    sample,
    color=['celltype'],
    cmap='turbo',
    title = '{} {} celltypes'.format(tissue_type, sample_name),
    show=False,            
)

ax = plt.gca() 
for cluster in sample.obs['leiden'].cat.categories:
    cluster_mask = sample.obs['leiden'] == cluster
    cluster_coords = sample.obsm['X_umap'][cluster_mask]
    x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
    ax.text(x, y, cluster, color='black', fontsize=10, weight='bold', ha='center', va='center')

plt.show()

In [ ]:
for group in ['ctr', 'hst', 'ftl']:
    adata_group = sample[sample.obs['group'] == group]
    
    sc.pl.umap(
    adata_group,
    color=['celltype'],
    title=f'{tissue_type} {sample_name} {group} celltypes',
    cmap='turbo', 
    show=False             
)

    ax = plt.gca()  
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10, weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
plt.rcParams.update({'font.size': 12})  

In [ ]:
sc.pl.dotplot(sample, markers, swap_axes=True, groupby='leiden', title = "{} {} Dotplot".format(tissue_type, sample_name), cmap='RdBu_r', vmin = -4, vmax = 4)

In [ ]:
sc.pl.stacked_violin(sample, markers, swap_axes=True, groupby='leiden', title = "{} {} Violinplot".format(tissue_type, sample_name), cmap='RdBu_r', vmin = -4, vmax = 4)